In [5]:
import pandas as pd
import numpy as np

from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

mysql_username = input(
    "MySQL username [root]: "
).strip() or "root"

mysql_password = getpass(
    "Enter your MySQL password: "
)

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_username,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="cart2insights"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

In [6]:
orders = pd.read_sql(
    "SELECT * FROM orders",
    engine,
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_sql(
    "SELECT * FROM order_items",
    engine
)

customers = pd.read_sql(
    "SELECT * FROM customers",
    engine
)

sellers = pd.read_sql(
    "SELECT * FROM sellers",
    engine
)

In [7]:
order_items["item_total"] = (
    order_items["price"] +
    order_items["freight_value"]
)

order_values = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        total_order_value=("item_total", "sum")
    )
)

order_features = orders.merge(
    order_values,
    on="order_id",
    how="left"
)

order_features["total_order_value"] = (
    order_features["total_order_value"]
    .fillna(0)
)

In [8]:
order_features["delivery_days"] = (
    order_features["order_delivered_customer_date"] -
    order_features["order_purchase_timestamp"]
).dt.days

In [9]:
on_time_condition = (
    (order_features["order_status"] == "delivered") &
    (
        order_features["order_delivered_customer_date"]
        <= order_features["order_estimated_delivery_date"]
    )
)

delayed_condition = (
    (order_features["order_status"] == "delivered") &
    (
        order_features["order_delivered_customer_date"]
        > order_features["order_estimated_delivery_date"]
    )
)

order_features["delivery_status"] = np.where(
    on_time_condition,
    "On Time",
    np.where(
        delayed_condition,
        "Delayed",
        "Not Delivered"
    )
)

In [10]:
order_features[
    [
        "order_id",
        "order_status",
        "total_order_value",
        "delivery_days",
        "delivery_status"
    ]
].head()

,order_id,order_status,total_order_value,delivery_days,delivery_status
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,72.19,7.0,On Time
1,00018f77f2f0320c557190d7a144bdd3,delivered,259.83,16.0,On Time
2,000229ec398224ef6ca0657da4fc703e,delivered,216.87,7.0,On Time
3,00024acbcdf0a6daa1e931b038114c75,delivered,25.78,6.0,On Time
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,218.04,25.0,On Time


In [11]:
customer_orders = order_features[
    order_features["order_status"] == "delivered"
].merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

In [16]:
customer_summary = (
    customer_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spending=(
            "total_order_value",
            "sum"
        )
    )
)

customer_summary["repeat_customer"] = np.where(
    customer_summary["customer_order_count"] > 1,
    "Yes",
    "No"
)

customer_summary.head()

,customer_unique_id,customer_order_count,customer_total_spending,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,No
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,No
2,0000f46a3911fa3c0805444483337064,1,86.22,No
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,No
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,No


In [13]:
delivered_order_ids = (
    orders.loc[
        orders["order_status"] == "delivered",
        ["order_id"]
    ]
    .drop_duplicates()
)

delivered_seller_items = order_items.merge(
    delivered_order_ids,
    on="order_id",
    how="inner"
)

seller_summary = (
    delivered_seller_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_revenue=("price", "sum"),
        seller_freight_value=("freight_value", "sum"),
        seller_gross_order_value=("item_total", "sum"),
        seller_order_count=("order_id", "nunique"),
        seller_items_sold=("order_item_id", "count")
    )
)

seller_summary = seller_summary.merge(
    sellers,
    on="seller_id",
    how="left"
)

seller_summary.head()

,seller_id,seller_revenue,seller_freight_value,seller_gross_order_value,seller_order_count,seller_items_sold,seller_zip_code_prefix,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,63.06,2748.06,3,3,09080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,24487.03,8655.87,33142.90,195,234,29156,cariacica,ES
2,002100f778ceb8431b7a1020ff7ab48f,1216.60,778.56,1995.16,50,54,14405,franca,SP
3,003554e2dce176b5555353e4f3555ac8,120.00,19.38,139.38,1,1,74565,goiania,GO
4,004c9cd9d87a3c30c522c48c4fc07416,19569.73,3524.29,23094.02,156,168,14940,ibitinga,SP


In [17]:
print(
    order_features[
        [
            "total_order_value",
            "delivery_days",
            "delivery_status"
        ]
    ].describe(include="all")
)

print("\nDelivery status:")
display(
    order_features["delivery_status"]
    .value_counts()
)

print("\nRepeat customers:")
display(
    customer_summary["repeat_customer"]
    .value_counts()
)

print("\nTop five sellers by revenue:")
display(
    seller_summary
    .sort_values("seller_revenue", ascending=False)
    .head()
)

        total_order_value  delivery_days delivery_status
count        99441.000000   96476.000000           99441
unique                NaN            NaN               3
top                   NaN            NaN         On Time
freq                  NaN            NaN           88644
mean           159.326166      12.094086             NaN
std            220.058804       9.551746             NaN
min              0.000000       0.000000             NaN
25%             61.050000       6.000000             NaN
50%            104.560000      10.000000             NaN
75%            176.000000      15.000000             NaN
max          13664.080000     209.000000             NaN

Delivery status:


delivery_status
On Time          88644
Delayed           7826
Not Delivered     2971
Name: count, dtype: int64


Repeat customers:


repeat_customer
No     90557
Yes     2801
Name: count, dtype: int64


Top five sellers by revenue:


,seller_id,seller_revenue,seller_freight_value,seller_gross_order_value,seller_order_count,seller_items_sold,seller_zip_code_prefix,seller_city,seller_state
834,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,20019.13,247007.06,1124,1148,14840,guariba,SP
982,53243585a1d6dc2643021fd1853d8905,217940.44,12856.58,230797.02,348,400,42738,lauro de freitas,BA
858,4a3ca9315b744ce9f8e9374361493884,196882.12,34338.31,231220.43,1772,1949,14940,ibitinga,SP
2903,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,9916.36,200833.50,578,579,13170,sumare,SP
1480,7c67e1448b00f6e969d365cea6b010ab,186570.05,51236.64,237806.69,973,1355,08577,itaquaquecetuba,SP


In [18]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

order_features.to_csv(
    processed_dir / "order_features.csv",
    index=False
)

customer_summary.to_csv(
    processed_dir / "customer_summary.csv",
    index=False
)

seller_summary.to_csv(
    processed_dir / "seller_summary.csv",
    index=False
)

print("Saved feature-engineered files to:")
print(processed_dir)

Saved feature-engineered files to:
c:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\processed


In [19]:
non_delivered_items = delivered_seller_items[
    ~delivered_seller_items["order_id"].isin(
        delivered_order_ids["order_id"]
    )
]

print(
    "Delivered seller items:",
    len(delivered_seller_items)
)

print(
    "Non-delivered items included:",
    len(non_delivered_items)
)

print(
    "Sellers summarized:",
    seller_summary["seller_id"].nunique()
)

Delivered seller items: 110197
Non-delivered items included: 0
Sellers summarized: 2970
